In [1]:
import pandas as pd
# Load powerplants.csv

df = pd.read_csv('powerplants.csv')

df.head()

,id,Name,Fueltype,Technology,Set,Country,Capacity,Efficiency,DateIn,DateRetrofit,DateOut,lat,lon,Duration,Volume_Mm3,DamHeight_m,StorageCapacity_MWh,EIC,projectID
0,0,Pumpspeicherkraftwerk Erzhausen,Hydro,Pumped Storage,Storage,Germany,200.0,0.75,1964.0,1998.0,NaN,51.898566,9.924887,5.510000,1.600000,287.0,1102.0,{nan},"{'MASTR': {'MASTR-SEE915985628661'}, 'GEM': {'..."
1,1,La Plate Taille,Hydro,Pumped Storage,Store,Belgium,144.0,NaN,1970.0,NaN,NaN,50.188400,4.386200,4.930556,68.400000,70.0,710.0,{nan},"{'GEM': {'G100000600151'}, 'JRC': {'JRC-H347'}..."
2,2,Illwerke Vkw Rodundwerk,Hydro,Reservoir,Store,Austria,495.0,0.75,1943.0,2011.0,NaN,47.085032,9.880116,588.383838,2.240000,353.0,291250.0,"{nan, nan, nan, nan, nan}","{'MASTR': {'MASTR-SEE952262880046', 'MASTR-SEE..."
3,3,Bissorte,Hydro,Pumped Storage,Store,France,818.0,NaN,1936.0,NaN,NaN,45.203600,6.581450,3.818182,39.500000,1160.0,3150.0,"{nan, nan}","{'GEM': {'G100001052026', 'G100000601707'}, 'J..."
4,4,Obervermuntwerk Maschine Turbine,Hydro,Pumped Storage,Storage,Austria,380.0,0.75,1943.0,2018.0,NaN,46.935290,10.059950,71.573684,35.630769,291.0,27198.0,"{nan, nan}","{'MASTR': {'MASTR-SEE926367113644', 'MASTR-SEE..."


In [2]:
# filter only danish powerplants
df_denmark = df[df['Country'] == 'Denmark']

# sum all capacities in Denmark
total_capacity_denmark = df_denmark['Capacity'].sum()
print(f"Total capacity in Denmark: {total_capacity_denmark} MW")

# find every unique technology in Denmark

unique_technologies = df_denmark['Technology'].unique()

print("Unique technologies in Denmark:")
for tech in unique_technologies:
    print(f" - {tech}")
    
# print every fueltype in Denmark

unique_fueltypes = df_denmark['Fueltype'].unique()

print("\nUnique fuel types in Denmark:")
for fuel in unique_fueltypes:
    print(f" - {fuel}")


Total capacity in Denmark: 17860.96002 MW
Unique technologies in Denmark:
 - OCGT
 - Combustion Engine
 - Steam Turbine
 - CCGT
 - Run-Of-River
 - nan
 - Onshore
 - Offshore
 - PV

Unique fuel types in Denmark:
 - Oil
 - Natural Gas
 - Solid Biomass
 - Biogas
 - Waste
 - Hydro
 - Hard Coal
 - Wind
 - Solar


In [3]:
import pypsa

network = pypsa.Network('base_s_3_elec_.nc')



INFO:pypsa.network.io:New version 1.3.0 available! (Current: 1.2.3)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, sub_networks


In [4]:
# sum all danish capacities in the network
tot_network = network.generators.loc[network.generators.bus.str.startswith('DK')]['p_nom_opt'].sum()
print(tot_network)

16960.94253


In [8]:
# Select generators in DK0 and DK1
dk_generators = network.generators[
    network.generators["bus"].str.startswith(("DK0", "DK1"))
]

# Exclude solar/PV, onshore wind and offshore wind
exclude = ["solar", "onwind", "offwind"]

dk_conventional = dk_generators[
    ~dk_generators.index.str.contains("|".join(exclude), case=False)
]

# Sum conventional capacity for each node
conventional_capacity_by_node = (
    dk_conventional.groupby("bus")["p_nom"].sum()
)

print(conventional_capacity_by_node)

bus
DK0 0AC    2198.760
DK1 0AC    3061.344
Name: p_nom, dtype: float64


In [6]:
# print the capacity of each carrier in Denmark
fuel_cap = network.generators.loc[network.generators.bus.str.startswith('DK')].groupby('carrier')['p_nom_opt'].sum()

print("\nCapacity of each carrier in Denmark:")
for carrier, capacity in fuel_cap.items():
    print(f" - {carrier}: {capacity} MW")


Capacity of each carrier in Denmark:
 - CCGT: 1629.83 MW
 - OCGT: 1049.602 MW
 - biomass: 1445.15 MW
 - coal: 385.0 MW
 - offwind-ac: 2565.40004 MW
 - offwind-dc: 0.0 MW
 - offwind-float: 0.0 MW
 - oil: 450.688 MW
 - onwind: 4147.761 MW
 - ror: 6.6339999999999995 MW
 - solar: 4987.67749 MW
 - solar-hsat: 0.0 MW
 - waste: 293.2 MW


In [7]:
# sum all 
5253

5253

In [12]:
# load 

df_test = pd.read_csv('ENS_wind_&_solar_register.csv', delimiter=";")

# Rows where postal code is missing
df_no_postcode = df_test[df_test["Postnr."].isna()]

# Total installed capacity where postal code is missing
df_no_postcode["InstalleretkW"].sum()

print("Number of installations without postal code:", len(df_no_postcode))
print("Installed capacity without postal code:", df_no_postcode["InstalleretkW"].sum(), "kW")

Number of installations without postal code: 0
Installed capacity without postal code: 0.0 kW
